In [ ]:
!pip -q uninstall tensorflow -y

In [ ]:
!cp -r /kaggle/input/lgelcm /kaggle/working/

In [ ]:
import os

os.chdir("/kaggle/working/lgelcm")
os.environ["WANDB_DISABLED"] = "true"

## INFERENCE

### Parameters
- **`--model_path`**: Base model directory.
- **`--adapter_path`**: LoRA adapter directory (also used by the tokenizer).
- **`--prompt_file`**: Text file with the system prompt.
- **`--test_input_glob`**: Glob that matches the test JSON files.
- **`--output_file`**: Output JSON path for aggregated results.
- **`--max_input_length`**: Max prompt length (tokens).
- **`--max_new_tokens`**: Max tokens to generate per sample.

In [ ]:
%%writefile /kaggle/working/lgelcm/eval/inference.py

import argparse
import json
import re
from glob import glob
from pathlib import Path
from typing import Iterable
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from accelerate import Accelerator
from accelerate.utils import gather_object

DEFAULT_MODEL_PATH = "/kaggle/input/datasets/iriscaius/mediphi-instrcut/MediPhi-Instruct"
DEFAULT_ADAPTER_PATH = (
    "/kaggle/input/datasets/iriscaius/"
    "lgelcm-lr2e-4-bs1-ga4-2026-04-24-08-04/checkpoint-1142"
 )
DEFAULT_PROMPT_FILE = "/kaggle/working/lgelcm/data/finetuning_prompt.txt"
DEFAULT_OUTPUT_FILE = "/kaggle/working/lgelcm/results/test_results.json"
DEFAULT_TEST_INPUT_GLOB = (
    "/kaggle/input/datasets/iriscaius/ct-chat-results/"
    "output_validation_llava_llama_3.1_8b.json"
 )
DEFAULT_MAX_INPUT_LENGTH = 8192
DEFAULT_MAX_NEW_TOKENS = 4096


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser("LoRA Finetuned LLM - Inference")
    parser.add_argument("--model_path", type=str, default=DEFAULT_MODEL_PATH)
    parser.add_argument("--adapter_path", type=str, default=DEFAULT_ADAPTER_PATH)
    parser.add_argument("--prompt_file", type=str, default=DEFAULT_PROMPT_FILE)
    parser.add_argument("--output_file", type=str, default=DEFAULT_OUTPUT_FILE)
    parser.add_argument("--test_input_glob", type=str, default=DEFAULT_TEST_INPUT_GLOB)
    parser.add_argument("--max_input_length", type=int, default=DEFAULT_MAX_INPUT_LENGTH)
    parser.add_argument("--max_new_tokens", type=int, default=DEFAULT_MAX_NEW_TOKENS)
    return parser.parse_args()


def read_text_file(path: str) -> str:
    return Path(path).read_text(encoding="utf-8").strip()


def load_json_file(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def load_tokenizer(adapter_path: str) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path,
        trust_remote_code=True,
        use_fast=True,
        padding="left",
    )
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model(
    model_path: str,
    adapter_path: str,
    tokenizer: AutoTokenizer,
    accelerator: Accelerator,
 ) -> PeftModel:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=torch.float16,
        device_map={"": accelerator.local_process_index},
        trust_remote_code=True,
    )
    base_model.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model


def load_system_prompt(prompt_file: str) -> str:
    return read_text_file(prompt_file)


def iter_test_samples(test_input_glob: str) -> Iterable[dict]:
    """
    Reads the new data format.
    Each record: { "id", "image", "ground_truth", "prediction" }
    """
    for file_path in glob(test_input_glob):
        data = load_json_file(file_path)
        for item in data:
            yield {
                "source_file": Path(file_path).name,
                "id": item.get("id", ""),
                "image": item.get("image", ""),
                "ground_truth": item["ground_truth"],
                "prediction": item["prediction"],
            }


def load_test_samples(test_input_glob: str) -> list[dict]:
    return list(iter_test_samples(test_input_glob))


def build_prompt(
    text: str,
    system_prompt: str,
    tokenizer: AutoTokenizer,
 ) -> str:
    """Builds a chat prompt for the given text (ground truth or prediction)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )


def generate_raw_text(
    prompt: str,
    model: PeftModel,
    tokenizer: AutoTokenizer,
    accelerator: Accelerator,
    max_input_length: int,
    max_new_tokens: int,
 ) -> str:
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(accelerator.device)

    with torch.no_grad():
        out_ids = model.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out_ids[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def _parse_json_list(raw_text: str) -> list:
    parsed = json.loads(raw_text)
    return parsed if isinstance(parsed, list) else []


def _extract_json_list_from_code_block(raw_text: str) -> list:
    match = re.search(r"```(?:json)?\s*(\[.*?\])\s*```", raw_text, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(1))
    except (json.JSONDecodeError, ValueError):
        return []


def parse_entities(raw_text: str) -> list:
    try:
        return _parse_json_list(raw_text)
    except (json.JSONDecodeError, ValueError):
        return _extract_json_list_from_code_block(raw_text)


def extract_entities_for_text(
    text: str,
    system_prompt: str,
    model: PeftModel,
    tokenizer: AutoTokenizer,
    accelerator: Accelerator,
    max_input_length: int,
    max_new_tokens: int,
 ) -> tuple[str, list]:
    """Builds a prompt, generates the output, and parses entities."""
    prompt = build_prompt(text, system_prompt, tokenizer)
    raw_text = generate_raw_text(
        prompt, model, tokenizer, accelerator, max_input_length, max_new_tokens
    )
    entities = parse_entities(raw_text)
    return raw_text, entities


def save_partial_results(results: list[dict], partial_path: Path, rank: int, idx: int) -> None:
    with open(partial_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    tqdm.write(f"[Rank {rank}] {idx} samples processed -> {partial_path}")


def run_inference_on_local_samples(
    local_samples: list[dict],
    system_prompt: str,
    model: PeftModel,
    tokenizer: AutoTokenizer,
    accelerator: Accelerator,
    max_input_length: int,
    max_new_tokens: int,
    output_file: str,
 ) -> list[dict]:
    local_results = []
    rank = accelerator.process_index

    output_dir = Path(output_file).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    partial_path = output_dir / f"rank_{rank}.json"

    iterator = tqdm(
        local_samples,
        desc=f"Inference (rank {rank})",
        disable=not accelerator.is_main_process,
    )

    for idx, sample in enumerate(iterator, 1):
        # --- Ground truth entity extraction ---
        gt_raw, gt_entities = extract_entities_for_text(
            text=sample["ground_truth"],
            system_prompt=system_prompt,
            model=model,
            tokenizer=tokenizer,
            accelerator=accelerator,
            max_input_length=max_input_length,
            max_new_tokens=max_new_tokens,
        )

        # --- Prediction entity extraction ---
        pred_raw, pred_entities = extract_entities_for_text(
            text=sample["prediction"],
            system_prompt=system_prompt,
            model=model,
            tokenizer=tokenizer,
            accelerator=accelerator,
            max_input_length=max_input_length,
            max_new_tokens=max_new_tokens,
        )

        local_results.append({
            "source_file": sample["source_file"],
            "id": sample["id"],
            "image": sample["image"],
            "ground_truth": sample["ground_truth"],
            "prediction": sample["prediction"],
            "gt_raw_output": gt_raw,
            "gt_entities": gt_entities,
            "pred_raw_output": pred_raw,
            "pred_entities": pred_entities,
        })

        if idx % 5 == 0:
            save_partial_results(local_results, partial_path, rank, idx)

    return local_results


def save_results(results: list[dict], output_file: str) -> None:
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"{len(results)} results saved -> {output_path}")


def main():
    args = parse_args()
    accelerator = Accelerator()

    tokenizer = load_tokenizer(args.adapter_path)
    model = load_model(args.model_path, args.adapter_path, tokenizer, accelerator)
    system_prompt = load_system_prompt(args.prompt_file)
    all_samples = load_test_samples(args.test_input_glob)

    if accelerator.is_main_process:
        print(f"Total samples  : {len(all_samples)}")
        print(f"Processes used : {accelerator.num_processes}")

    with accelerator.split_between_processes(all_samples) as local_samples:
        local_results = run_inference_on_local_samples(
            local_samples=local_samples,
            system_prompt=system_prompt,
            model=model,
            tokenizer=tokenizer,
            accelerator=accelerator,
            max_input_length=args.max_input_length,
            max_new_tokens=args.max_new_tokens,
            output_file=args.output_file,
        )

    all_results = gather_object(local_results)

    if accelerator.is_main_process:
        save_results(all_results, args.output_file)


if __name__ == "__main__":
    main()

In [ ]:
!accelerate launch --num_processes 2 -m eval.inference --adapter_path /kaggle/input/datasets/iriscaius/lgelcm-lr2e-4-bs1-ga4-2026-04-24-08-04/checkpoint-1142